In [60]:
import pandas as pd
from yaml import safe_load

from rfm_sp500 import constants

with open(constants.CONFIG_FILE, "r") as f:
    config = safe_load(f)

start_date = config["start_date"]
end_date_exclusive = config["end_date_exclusive"]

In [61]:
sp_dir = constants.RAW_DATA_DIR / f"sp500_data_{start_date}_{end_date_exclusive}.csv.gz"
sp500_df = pd.read_csv(sp_dir)
empty_tickers = ["FDXF", "HONA", "Q", "SNDK", "VYLR"]
sp500_df = sp500_df[~sp500_df["Ticker"].isin(empty_tickers)]
sp500_df.head()

,Date,Ticker,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
0,2021-01-04,A,114.009102,118.639999,0.194,120.089996,117.110001,118.940002,0.0,2030700.0
1,2021-01-05,A,114.941277,119.610001,0.000,120.320000,118.070000,118.480003,0.0,2344000.0
2,2021-01-06,A,118.093231,122.889999,0.000,123.879997,118.209999,118.690002,0.0,2428500.0
3,2021-01-07,A,121.235580,126.160004,0.000,127.480003,124.110001,124.330002,0.0,1775900.0
4,2021-01-08,A,122.100464,127.059998,0.000,127.639999,125.849998,126.489998,0.0,1577200.0


In [62]:
# Check for duplicates

duplicate_ticker_date = sp500_df[sp500_df.duplicated(subset=["Ticker", "Date"], keep=False)]
duplicate_ticker_date

,Date,Ticker,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume


No duplicates

In [63]:
tickers = sp500_df["Ticker"].unique()

missing_in_middle = []
inconsistent_leading = []
inconsistent_trailing = []

for ticker in tickers:
    df = sp500_df[sp500_df["Ticker"] == ticker]
    df = df.drop(columns=["Ticker", "Date"])
    leading_counts = {}
    trailing_counts = {}

    for col in df.columns:
        first = df[col].first_valid_index()
        last = df[col].last_valid_index()

        middle = df[col].loc[first:last].isna()
        trailing = df[col].loc[last:].isna()
        leading = df[col].loc[:first].isna()

        leading_counts[col] = leading.sum()
        trailing_counts[col] = trailing.sum()

        if middle.sum() > 0:
            missing_in_middle.append((ticker, col))
            print(
                f"Missing values in the middle for ticker: {ticker}, column: {col},"
                f"count: {middle.sum()}"
            )

    if len(set(leading_counts.values())) > 1:
        print(f"Leading NaN counts differ for ticker: {ticker}")
        inconsistent_leading.append(ticker)

    if len(set(trailing_counts.values())) > 1:
        print(f"Trailing NaN counts differ for ticker: {ticker}")
        inconsistent_trailing.append(ticker)

There is no data missing throughout available historical data other than leading and trailing NaNs and both leading and trailing NaNs are the same for each column

Since everything is consistent with this snapshot of dataset can continue 

In [64]:
benchmark_dir = constants.RAW_DATA_DIR / f"benchmark_data_{start_date}_{end_date_exclusive}.csv.gz"
benchmark_df = pd.read_csv(benchmark_dir)
benchmark_df.head()

,Date,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
0,2021-01-04,3700.649902,3700.649902,0.0,3769.989990,3662.709961,3764.610107,0.0,5015000000
1,2021-01-05,3726.860107,3726.860107,0.0,3737.830078,3695.070068,3698.020020,0.0,4591020000
2,2021-01-06,3748.139893,3748.139893,0.0,3783.040039,3705.340088,3712.199951,0.0,6064110000
3,2021-01-07,3803.790039,3803.790039,0.0,3811.550049,3764.709961,3764.709961,0.0,5099160000
4,2021-01-08,3824.679932,3824.679932,0.0,3826.689941,3783.600098,3815.050049,0.0,4773040000


In [65]:
benchmark_duplicate_ticker_date = benchmark_df[benchmark_df.duplicated(subset=["Date"], keep=False)]
benchmark_duplicate_ticker_date

,Date,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume


In [66]:
missing_in_middle = []
leading_counts = {}
trailing_counts = {}

for col in benchmark_df.columns:
    first = benchmark_df[col].first_valid_index()
    last = benchmark_df[col].last_valid_index()

    middle = benchmark_df[col].loc[first:last].isna()
    trailing = benchmark_df[col].loc[last:].isna()
    leading = benchmark_df[col].loc[:first].isna()

    leading_counts[col] = leading.sum()
    trailing_counts[col] = trailing.sum()

    if middle.sum() > 0:
        missing_in_middle.append((col))
        print(f"Missing values in the middle for column: {col},count: {middle.sum()}")

    if len(set(leading_counts.values())) > 1:
        print("Leading NaN counts differ")

    if len(set(trailing_counts.values())) > 1:
        print("Trailing NaN counts differ")